# Solución Ejercicio 02c  |  Tema 4 - PRyES
## Intervalos de confianza
stats

In [ ]:
# import required packages
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd

### Sea una v.a. X que se distribuye como una Normal, de la que se dispone de la muestra almacenada en el fichero muestra_n100v01y2022.csv de tamaño n que se guarda en la variable x con el siguiente código: 

In [ ]:
# cargamos los datos de la muestra disponible

df_orig = pd.read_csv('muestra_n100v01y2022.csv')
print('Tamaño de df_orig con los datos cargados:', df_orig.shape)

# datos de la muestra para trabajar
x = df_orig.x.tolist()    #   <-------------------------------- x (valores en la muestra)
n = len(x)                #   <-------------------------------- n (tamaño de la muestra)

# observamos su distribución
plt.figure
plt.hist(x);plt.xlabel('x');plt.title(f'Muestra (n={n})')
plt.show()

### a) Calcula un intervalo de confianza para la varianza poblacional con un nivel de confianza de 95%. Explica claramente los pasos seguidos para crearlo.

In [ ]:
# Respuesta

# calculamos la desviación típica muestral sx sin sesgo (dividiendo por n-1)
sx = np.std(x, ddof=1)
CHI = stats.chi2(n-1)

gamma = 0.95
alfa = 1 - gamma
val_crit_alfa_medios = CHI.ppf(alfa/2)
val_crit_uno_menos_alfa_medios = CHI.ppf(1 - (alfa/2))

lim_inf_var = (n-1) * sx**2 / val_crit_uno_menos_alfa_medios
lim_sup_var = (n-1) * sx**2 / val_crit_alfa_medios

print(f'El intervalo al {100*gamma}% de la varianza de X es: [{lim_inf_var}, {lim_sup_var}]')

### b) Representa en la misma gráfica, la función de densidad de la distribución de probabilidad utilizada para calcular los dos valores críticos necesarios para calcular el intervalo de confianza de la varianza poblacional. Añade en la misma dichos valores críticos e indica claramente el área asociada al nivel de confianza utilizado. Incluye también el valor medio de la distribución para ver la posible simetría.

In [ ]:
# Respuesta

# puntos para representar la distribución y el intervalo
xs = np.linspace(0, CHI.ppf(0.999), 1000)
xIC = np.linspace(val_crit_alfa_medios, val_crit_uno_menos_alfa_medios, 1000)

plt.figure(1, figsize=(15, 6))
plt.plot(xs, CHI.pdf(xs), linewidth = 4, label=f'Distribución chi2')
plt.fill_between(xIC, CHI.pdf(xIC), color="green", alpha=0.3, label=f'Probabilidad asociada ($\gamma=${gamma})')

# valores críticos
plt.plot(val_crit_alfa_medios, 0, 'o', markersize = 10, label=f'Valor crítico izquierda (quantil {alfa/2})')
plt.plot(val_crit_uno_menos_alfa_medios, 0, 'o', markersize = 10, label=f'Valor crítico derecha (quantil {1-alfa/2})')

# media
media_chi = CHI.mean()
plt.vlines(media_chi, 0, np.max(CHI.pdf(xs)), color='black', label=f'Media Chi2 ({media_chi})')


plt.title('Distribución de la chi-cuadrado con n-1 grados de libertad')
plt.tight_layout()
plt.legend()
plt.show()

### c) Calcula ahora un intervalo de confianza al 95% para la desviación típica poblacional de X sin realizar ningún tipo de suposición sobre la distribución de dicha v.a.

In [ ]:
# Respuesta

print('Utilizamos Bootstrap para calcular dicho intervalo')
print('En este caso el estadístico será la desviación típica')
# params para llamar a bootstrap (método BCa, el que se suele usar normalmente)
muestra_orig = (x,)  # tiene que ser una secuencia
R = 9999
estadistico = np.std
nivel_confianza = 0.95

res_BCa = stats.bootstrap(muestra_orig, estadistico, confidence_level = nivel_confianza,  n_resamples = R, method = 'BCA', random_state = 999)

# devuelve como resultado el IC y el SE del estadístico
lim_inf_BCa, lim_sup_BCa = res_BCa.confidence_interval
SE_BCa = res_BCa.standard_error

print(f'IC {100*nivel_confianza}% estimado según método BCa para la desviación típica muestral: [{lim_inf_BCa}, {lim_sup_BCa}]')
print(f'Error estándar del estimador desviación típica muestral, estimado según método BCa: {SE_BCa}')

### d) Compara el intervalo de confianza anterior con el que se puede derivar del intervalo de confianza estimado previamente para la varianza poblacional suponiendo que X sigue una distribución normal. ¿Es muy diferente? ¿Qué implica entonces?

In [ ]:
# Respuesta

print(f'El intervalo obtenido al 95% de la varianza poblacional de X, obtenido suponiendo que X se distribuye como una normal es : [{lim_inf_var}, {lim_sup_var}]')
print('Si tomamos la raíz cuadrada a ambos límites podemos obtener el intervalo a ese nivel de confianza para la desviación típica:')
lim_inf_std =  np.sqrt(lim_inf_var)
lim_sup_std = np.sqrt(lim_sup_var)

print(f'Por tanto, el intervalo obtenido al 95% de la std poblacional de X es: [{lim_inf_std}, {lim_sup_std}]')

print(f'Si se compara con el intervalo obtenido al 95% sin suponer que X sigue una normal, utilizando bootstrap: [{lim_inf_BCa}, {lim_sup_BCa}]')
print('Se puede ver no hay mucha diferencia. Si X sigue realmente una normal y el tamaño muestral se fuese aumentando, es de esperar que se parecieran cada vez más.')


In [ ]:
# gráficamente se pueden mostrar ambos intervalos

# puntos para representar la distribución y el intervalo
xICteo = np.linspace(lim_inf_std, lim_sup_std, 1000)
xICboots = np.linspace(lim_inf_BCa, lim_sup_BCa, 1000)


plt.figure(1, figsize=(15, 6))
plt.hlines(-0.7, lim_inf_std, lim_sup_std,  color = 'darkgreen', linewidth = 4, label = f'intervalo X es normal [{lim_inf_var}, {lim_sup_var}]')
plt.hlines(-0.8, lim_inf_BCa, lim_sup_BCa,  color = 'red', linewidth = 4, label = f'intervalo Bootstrap [{lim_inf_var}, {lim_sup_var}]')

plt.vlines(sx, -0.8, 1, color='black', label=f'Estimación puntual varianza {sx**2}')

plt.title('IC asumiendo que X es normal y sin asumir nada (con bootstrap)')
plt.xlabel('Desviación típica de x')
plt.tight_layout()
plt.legend()
plt.show()


print('Podría ser que parte de las diferencias se deban al cálculo de la desviación típica, ya que usando bootstrap hemos '
      'usado la versión por defecto de std, que divide por n, mientras que en el cálculo "teórico" estamos utilizando la versión dividiendo '
      ' por n-1.')